# Advanced Business Analytics — Main Report

**Course:** Advanced Business Analytics  
**Institution:** Technical University of Denmark (DTU)  
**Group:** XXX  
**Authors:** Simone Egede von Mehren s190739, XXX, XXX, XXX 

---

## Abstract

*(Write 4–6 sentences summarising the research question, data, methods, and key finding once the analysis is complete.)*

## Table of Contents

1. [Research Question](#1-research-question)
2. [Data](#2-data)
3. [Feature Engineering](#3-feature-engineering)
4. [Modelling](#4-modelling)
5. [Explainability and Endogeneity](#5-explainability-and-endogeneity)
6. [Results](#6-results)
7. [Discussion](#7-discussion)
8. [Conclusion](#8-conclusion)
9. [Appendix](#9-appendix)

---
## 1. Research Question

*(State the research question and motivation. What are we trying to predict and why does it matter?)*

---
## 2. Data

We use three main data sources, each covering the period 14 November 2025 – 10 April 2026 (148 days):

| Source | Description |
|---|---|
| Truth Social posts | 1,900+ posts scraped from @realDonaldTrump |
| News articles | XXX |
| Macro indicators | XXX |

All sources are merged to a single daily-frequency dataset of 148 trading days (`joined_features/full_dataset.csv`). Full collection and cleaning code is provided in the [Appendix](#9-appendix).

### 2.1 Data collection

**Truth Social posts:** *(Describe how posts were scraped — tool/API used, scrape dates, any rate limits or gaps.)*

**News data:** *(Describe source, query terms, date range, and any filtering applied.)*

**Macro indicators:** *(Describe how FRED data was downloaded — API or manual export, series IDs, frequency.)*

---
## 3. Feature Engineering

### 3.1 Trump post features — behavioural

Raw data consists of 3 JSON scrape files from @realDonaldTrump on Truth Social, covering 14 November 2025 – 10 April 2026. After deduplication (keeping the most recently scraped copy of each post so engagement counts are as current as possible), the dataset contains 1,939 posts across 146 posting days. Two calendar days in the period have no posts (2025-12-21 and 2025-12-29) and are retained in the daily dataset with zero-filled engagement counts.

Posts containing only images and no text are flagged separately, as they cannot contribute to NLP features. The following 11 behavioural signals are computed at daily frequency:

| Feature | Description |
|---|---|
| `post_count` | Total number of posts in the day |
| `n_image_only` | Number of posts with no text content |
| `avg_upvotes` | Mean upvote count across posts |
| `max_upvotes` | Maximum upvote count (peak engagement) |
| `total_upvotes` | Sum of upvotes — captures total daily reach |
| `avg_reblogs` | Mean reblog (reshare) count |
| `total_reblogs` | Sum of reblogs |
| `avg_replies` | Mean reply count |
| `n_quotes` | Number of posts that quote another post |
| `n_with_link` | Number of posts containing an external link |
| `avg_text_len` | Average character length of text posts |

Follower count was extracted from the raw data but found to take only two distinct levels (~11.97M and ~12.32M) reflecting the two scrape batches rather than genuine daily variation. It is therefore excluded from the feature set.
Full code is provided in **Appendix A** (`appendix_A_trump/01_trump_data.ipynb`).

### 3.2 Trump post features — NLP

#### Text preprocessing

Before any model is applied, post text is cleaned in three steps: (1) URL removal, including Truth Social's broken URL format where spaces are inserted mid-link; (2) HTML tag stripping; (3) normalisation of whitespace and removal of leading/trailing spaces.

#### Topic modelling (BERTopic)

Topic modelling is performed using BERTopic. Each post is encoded into a high-dimensional embedding using a pre-trained sentence transformer (`all-MiniLM-L6-v2`). Embeddings are reduced to 5 dimensions with UMAP and clustered with HDBSCAN. Topic labels are derived from class-based TF-IDF over the words in each cluster. A fixed random seed is used for reproducibility.

BERTopic identifies 13 topics plus an outlier class (topic -1) for posts that do not fit any coherent group. The table below lists all topics with their representative keywords and whether they are retained as predictive signals:

| Topic | Keywords | Posts | Used | Rationale |
|---|---|---|---|---|
| 0 | endorsement, state, secure, district | 267 | No | Formulaic endorsement posts — no market mechanism |
| 1 | tariffs, united states, court | 152 | **Yes** | Strongest economic signal |
| 2 | minnesota, ICE, criminals, democrats | 134 | **Yes** | Immigration — political resilience signal |
| 3 | Iran, president donald | 114 | **Yes** | Foreign policy / geopolitical risk |
| 4 | white house, ballroom, center | 69 | No | Event announcements — no interpretable signal |
| 5 | president DJT, great, fantastic | 46 | No | Generic presidential messaging |
| 6 | Epstein, fake news, New York Times | 43 | **Yes** | Media attacks — base-rallying signal |
| 7 | Venezuela, oil, Honduras | 42 | **Yes** | Energy / sanctions signal |
| 8 | state of the union, address | 31 | No | Single recurring event, no daily variance |
| 9 | Save America, voter ID | 28 | **Yes** | Election integrity signal |
| 10 | peace, Gaza, NATO | 26 | **Yes** | Geopolitical uncertainty |
| 11 | judge, court, district | 23 | **Yes** | Judicial / policy uncertainty |
| 12 | water, Mexico, storm, environmental | 21 | No | Only 21 posts — too sparse |

For each post, BERTopic returns a probability score for every topic. These are summed (not averaged) across all posts in a day to form the daily `topic_activity` signals. Summing preserves posting intensity — a day with 20 tariff posts is meaningfully different from a day with 2 — and avoids perfect multicollinearity that would arise if per-post probability vectors (which sum to 1) were averaged.

#### Sentiment analysis (VADER)

VADER (Valence Aware Dictionary and sEntiment Reasoner) is applied to each cleaned post, producing a compound sentiment score in [-1, 1]. Three daily signals are derived:

| Feature | Description |
|---|---|
| `sentiment_mean` | Mean compound score across all posts in the day |
| `sentiment_std` | Standard deviation — captures within-day tonal variation |
| `sentiment_pct_negative` | Share of posts with compound score < -0.05 |

A spot-check of extreme days confirms that the most negative and positive days tend to be single-post days where one unusually toned post dominates the average. On high-volume days, the daily score reflects an average across a full news cycle. A notable limitation is that VADER reads adversarial posts (criminal referrals, attacks on opponents) as negative regardless of political intent, which may not translate to a negative market signal.

Full code is provided in **Appendix A** (`appendix_A_trump/02_trump_nlp.ipynb`).

### 3.3 News sentiment features

*(Add description of news data source, collection method, and how daily sentiment signals are derived. Include a feature table analogous to 3.1 and 3.2.)*

Full code is provided in **Appendix B** (`appendix_B_news/`).

### 3.4 Macro features

*(Add description of FRED indicators used (CPI, Fed Funds Rate, Unemployment, Consumer Sentiment), any transformations applied (e.g. log differences, lags), and a feature table.)*

Full code is provided in **Appendix C** (`appendix_C_macro/`).

---
## 4. Modelling

*(Describe the model(s) used, train/test split, evaluation metric, and any regularisation choices.)*

Full code is provided in **Appendix D** (`appendix_D_modelling/`).

---
## 5. Explainability and Endogeneity

### 5.1 Explainable AI

*(Describe which explainability methods are applied — e.g. SHAP values, permutation importance, partial dependence plots. Which features drive predictions and in which direction?)*

### 5.2 Endogeneity

A key concern in this analysis is endogeneity — the possibility that the relationship between Trump's posting behaviour and the outcome variable runs in both directions, or that both are driven by a common third factor.

**Reverse causality:** Trump may post more (or more aggressively) in response to market or political developments, rather than causing them. To mitigate this, all NLP and behavioural features are lagged by at least one day so that post content strictly precedes the outcome variable.

**Omitted variable bias:** External shocks — geopolitical events, Fed announcements, earnings releases — may simultaneously affect both Trump's posting behaviour and the outcome. Macro indicators (CPI, Fed Funds Rate, Unemployment, Consumer Sentiment) are included as controls to absorb part of this variation.

*(Describe any additional steps taken — instrumental variables, robustness checks, placebo tests — and discuss remaining limitations.)*

---
## 6. Results

*(Key results — tables, figures. Pull in the most important outputs from the analysis notebooks.)*

---
## 7. Discussion

*(Interpret results, limitations, alternative explanations.)*

---
## 8. Conclusion

*(2–4 sentences answering the research question and pointing to future work.)*

---
## 9. Appendix

### Appendix A — Trump Truth Social data

| Notebook | Contents |
|---|---|
| `appendix_A_trump/01_trump_data.ipynb` | Raw data loading, cleaning, behavioural feature engineering, daily aggregation |
| `appendix_A_trump/02_trump_nlp.ipynb` | Text preprocessing, BERTopic topic modelling, VADER sentiment analysis, NLP feature export |

### Appendix B — News data

| Notebook / Script | Contents |
|---|---|
| *(XXX)* | *(XXX)* |

### Appendix C — Macro indicators

| Notebook / Script | Contents |
|---|---|
| *(XXX)* | *(XXX)* |

### Appendix D — Modelling

| Notebook | Contents |
|---|---|
| *(XXX)* | *(XXX)* |